# Phase 7: Reverse Distillation, configuration C8, product vial

**CRISP-DM stage:** Modelling

**The idea, with the vial inspector:** a teacher (Wide ResNet-50, pretrained on ImageNet and never changed) describes each photo as layers of features. A student (anomalib's "bottleneck" and "decoder") gets only a squeezed summary of the teacher's description and must rebuild the full description. It practises only on good parts, so it learns to rebuild good parts well. On a defect it has never seen, its rebuild goes wrong, and where it goes wrong is the heat map.

**Training words:**
- **Loss:** how badly the student's rebuild matches the teacher, measured as 1 minus the cosine similarity for each layer, summed over the three layers. 0 would be a perfect copy.
- **Optimizer:** Adam, the rule that nudges the student's numbers to lower the loss after each batch. Learning rate 0.005 and betas (0.5, 0.99) are anomalib's fixed settings.
- **Epoch:** one pass through every good training photo. This run trains for the number of epochs in `configs/C8_*.yaml` (200, decided in Phase 7).

**Two notebooks:** `07_core_C8_sheet_metal` and `07_core_C8_vial` are identical except for the product below, so the two can train at the same time on Kaggle.

**Saved as it goes:** the loss after every epoch, and the student's weights every 25 epochs, go to Kaggle's output, so a disconnect cannot lose the work. The weights are a fitted model, so they never go into git.

**Measuring** is the same as for every other method: benchmark threshold from good validation photos, half A scored with `src/evaluate.py`. **Half B is never read.**

**Runs on:** Kaggle with the GPU and internet on. On the laptop it only checks the code, with an untrained teacher, tiny photos, 1 epoch and the sample folder.

In [ ]:
!pip install anomalib==2.7.0

In [ ]:
CONFIG_ID = "C8"
PRODUCT = "vial"

import csv
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy
import torch
import yaml
from anomalib.data import MVTecAD2
from anomalib.models import ReverseDistillation
from anomalib.models.image.reverse_distillation.loss import ReverseDistillationLoss
from anomalib.models.image.reverse_distillation.torch_model import ReverseDistillationModel
from torch.utils.data import DataLoader, Subset

USING_GPU = torch.cuda.is_available()
if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    PRETRAINED = True
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 54, "vial": 70}
else:
    # The laptop only proves the code runs
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    PRETRAINED = False
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 2, "vial": 4}

code_folder = None
if USING_GPU:
    for evaluate_file in Path("/kaggle/input").rglob("evaluate.py"):
        code_folder = evaluate_file.parent.parent
else:
    code_folder = Path(".")
if code_folder is None:
    raise FileNotFoundError("Could not find src/evaluate.py")
sys.path.insert(0, str(code_folder))
from src import evaluate

config_files = list((code_folder / "configs").glob(CONFIG_ID + "_*.yaml"))
with open(config_files[0], encoding="utf-8") as file:
    config = yaml.safe_load(file)
method = config["method"]
# Stops here if the settings on Kaggle are an old copy without the decided length
if not isinstance(method["training_length"], int):
    raise ValueError("training_length is not decided in this settings file: " + str(method["training_length"]))
if USING_GPU:
    EPOCHS = method["training_length"]
    IMAGE_SIZE = config["image"]["size"]
else:
    EPOCHS = 1
    IMAGE_SIZE = 64
BATCH_SIZE = method["train_batch_size"]
SEED = config["seeds"][0]
SAVE_EVERY = 25
print("Settings:", config_files[0].name, "| product:", PRODUCT, "| epochs:", EPOCHS, "| batch size:", BATCH_SIZE,
      "| learning rate:", method["learning_rate"], "| betas:", method["betas"], "| seed:", SEED)

HALF_A_PARTS = {
    "sheet_metal": {"good": ["001", "002"], "bad": ["000", "004", "005", "007", "009", "012", "013"]},
    "vial": {"good": ["000", "002", "003"], "bad": ["001", "008", "010", "011", "012", "013", "014"]},
}
HEAT_MAP_PARTS = {
    "sheet_metal": [("bad", "000"), ("bad", "004"), ("bad", "009"), ("good", "001")],
    "vial": [("bad", "001"), ("bad", "010"), ("bad", "011"), ("good", "000")],
}
RESULTS_FOLDER = Path("results")
RESULTS_FOLDER.mkdir(exist_ok=True)
RUN_NAME = CONFIG_ID + "_" + PRODUCT + "_seed" + str(SEED)
recipe = ReverseDistillation.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform


def set_seed(seed):
    """Fix the random numbers used by Python, NumPy and PyTorch."""
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)


def make_loader(dataset, positions=None):
    """Build batches in a fixed order, optionally from chosen photos only."""
    source = dataset
    if positions is not None:
        source = Subset(dataset, positions)
    return DataLoader(source, batch_size=BATCH_SIZE, collate_fn=dataset.collate_fn, shuffle=False)


def score_photos(model, loader):
    """Run the model on photos. Return maps, scores, masks, paths and model seconds."""
    maps = []
    scores = []
    masks = []
    paths = []
    model_seconds = 0.0
    with torch.no_grad():
        for batch in loader:
            images, batch_masks = recipe(batch.image, batch.gt_mask)
            images = images.to(DEVICE)
            started = evaluate.start_timer()
            output = model(images)
            model_seconds = model_seconds + evaluate.stop_timer(started)
            # The map has an extra dimension for its one channel
            maps.append(output.anomaly_map.squeeze(1).cpu())
            # Scores come as a column, shaped [photos, 1], so they are turned into a simple list
            scores.append(output.pred_score.reshape(-1).cpu())
            masks.append(batch_masks)
            paths.extend(batch.image_path)
    return torch.cat(maps), torch.cat(scores), torch.cat(masks), paths, model_seconds

## Train the student

The loss is printed every 25 epochs, when the student is saved.

In [ ]:
product_folder = None
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name == PRODUCT:
        product_folder = train_folder.parent
datamodule = MVTecAD2(root=product_folder.parent, category=PRODUCT, num_workers=0)
datamodule.setup()

set_seed(SEED)
model = ReverseDistillationModel(backbone=method["backbone"], input_size=(IMAGE_SIZE, IMAGE_SIZE),
                                 layers=method["layers"], anomaly_map_mode=method["anomaly_map_mode"],
                                 pre_trained=PRETRAINED).to(DEVICE)
# Only the student (bottleneck and decoder) learns. The teacher (encoder) stays frozen
student_parameters = list(model.decoder.parameters()) + list(model.bottleneck.parameters())
optimizer = torch.optim.Adam(student_parameters, lr=method["learning_rate"], betas=tuple(method["betas"]))
loss_function = ReverseDistillationLoss()
shuffler = torch.Generator()
shuffler.manual_seed(SEED)
train_loader = DataLoader(datamodule.train_data, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=datamodule.train_data.collate_fn, generator=shuffler)

loss_rows = []
evaluate.reset_peak_memory()
training_started = evaluate.start_timer()
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_total = 0.0
    batches = 0
    for batch in train_loader:
        images = recipe(batch.image).to(DEVICE)
        teacher_features, student_features = model(images)
        loss = loss_function(teacher_features, student_features)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_total = loss_total + loss.item()
        batches = batches + 1
    loss_rows.append([epoch, round(loss_total / batches, 6)])
    # Saved every epoch, so a disconnect cannot lose the loss history
    with open(RESULTS_FOLDER / (RUN_NAME + "_loss.csv"), "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["epoch", "average_loss"])
        for row in loss_rows:
            writer.writerow(row)
    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        torch.save({"epoch": epoch, "bottleneck": model.bottleneck.state_dict(), "decoder": model.decoder.state_dict()},
                   RUN_NAME + "_student.pt")
        print("epoch", epoch, "average loss", loss_rows[-1][1], "| student saved")
fit_seconds = evaluate.stop_timer(training_started)
peak_fit = evaluate.peak_memory_gb()
print("Training took", round(fit_seconds / 60, 1), "minutes, peak", peak_fit, "GB")

epochs_done = []
losses = []
for row in loss_rows:
    epochs_done.append(row[0])
    losses.append(row[1])
figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(epochs_done, losses)
axis.set_xlabel("Epoch")
axis.set_ylabel("Average loss (lower is a closer copy)")
axis.set_title("Reverse Distillation training loss, " + PRODUCT + ", seed " + str(SEED))
axis.grid(alpha=0.3)
figure.tight_layout()
figure.savefig(RESULTS_FOLDER / ("phase7_rd_loss_" + PRODUCT + ".png"), dpi=150)
plt.close(figure)

## Measure on half A

The same steps as every other method.

In [ ]:
model.eval()
with torch.no_grad():
    model(torch.zeros(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))

half_a_positions = []
test_paths = datamodule.test_public_data.samples["image_path"].tolist()
for position in range(len(test_paths)):
    path = Path(test_paths[position])
    if path.stem.split("_")[0] in HALF_A_PARTS[PRODUCT][path.parent.name]:
        half_a_positions.append(position)
if len(half_a_positions) != EXPECTED_HALF_A_PHOTOS[PRODUCT]:
    raise ValueError("Wrong number of half A photos: " + str(len(half_a_positions)))

validation_maps, validation_scores, ignored_masks, ignored_paths, ignored_seconds = score_photos(
    model, make_loader(datamodule.val_data))
evaluate.reset_peak_memory()
maps, scores, masks, paths, model_seconds = score_photos(
    model, make_loader(datamodule.test_public_data, half_a_positions))
peak_scoring = evaluate.peak_memory_gb()

labels = []
for path in paths:
    labels.append(Path(path).parent.name == "bad")
labels = torch.tensor(labels)

results = evaluate.evaluate(validation_maps, maps, scores, masks, labels, model_seconds, peak_scoring)
threshold = results["threshold"]
results["config"] = CONFIG_ID
results["product"] = PRODUCT
results["seed"] = SEED
results["half_a_photos"] = len(paths)
results["defective_photos"] = int(labels.sum())
results["good_validation_photos_flagged"] = int((validation_scores > threshold).sum())
results["good_validation_photos"] = len(validation_scores)
results["good_half_a_photos_flagged"] = int(((scores > threshold) & ~labels).sum())
results["defective_half_a_photos_flagged"] = int(((scores > threshold) & labels).sum())
results["fit_seconds"] = round(fit_seconds, 2)
results["peak_gpu_gb_fit"] = peak_fit
results["epochs"] = EPOCHS
results["final_loss"] = loss_rows[-1][1]
results["pretrained"] = PRETRAINED
results["image_size"] = IMAGE_SIZE

with open(RESULTS_FOLDER / (RUN_NAME + "_metrics.json"), "w") as file:
    json.dump(results, file, indent=1)
with open(RESULTS_FOLDER / (RUN_NAME + "_photo_scores.csv"), "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["file_name", "label", "score", "above_threshold"])
    for position in range(len(paths)):
        writer.writerow([Path(paths[position]).name, Path(paths[position]).parent.name,
                         round(float(scores[position]), 6), bool(scores[position] > threshold)])
for key in results:
    print(" ", key, results[key])

## Heat maps and failure gallery

In [ ]:
normalize_step = recipe.transforms[1]
means = torch.tensor(normalize_step.mean).view(3, 1, 1)
stds = torch.tensor(normalize_step.std).view(3, 1, 1)
top = float(maps.max())
bottom = float(maps.min())


def draw_gallery(picked, file_name, title):
    """Draw each picked photo with its heat map, mask outline and score, then save and close."""
    figure, axes = plt.subplots(len(picked), 2, figsize=(8, 3.6 * len(picked)), squeeze=False)
    for row_number in range(len(picked)):
        kind, position = picked[row_number]
        path = Path(paths[position])
        photo = recipe(datamodule.test_public_data[half_a_positions[position]].image)
        photo = (photo * stds + means).clamp(0, 1)
        mask = masks[position].float()
        axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
        axes[row_number][1].imshow(maps[position], cmap="Blues", vmin=bottom, vmax=top)
        if bool(mask.any()):
            axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        axes[row_number][0].set_title(kind + ": " + path.parent.name + " " + path.name, fontsize=9)
        axes[row_number][1].set_title("score " + str(round(float(scores[position]), 3)), fontsize=9)
        axes[row_number][0].axis("off")
        axes[row_number][1].axis("off")
    figure.suptitle(title, y=1.0)
    figure.tight_layout()
    figure.savefig(file_name, dpi=120, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)


# Heat maps for the same fixed photos as in Phase 5, chosen in advance
fixed = []
for position in range(len(paths)):
    path = Path(paths[position])
    for label, part in HEAT_MAP_PARTS[PRODUCT]:
        if path.parent.name == label and path.name == part + "_regular.png":
            fixed.append(("fixed", position))
if len(fixed) > 0:
    draw_gallery(fixed, "gallery_heatmaps_" + RUN_NAME + ".png",
                 "MVTec AD 2, " + PRODUCT + " (CC BY-NC-SA 4.0), half A, " + CONFIG_ID + ". Red: mask")

# Failure gallery, picked by score as in Phase 6
defective = []
good = []
for position in range(len(paths)):
    if labels[position]:
        defective.append(position)
    else:
        good.append(position)


def score_of(position):
    return float(scores[position])


defective.sort(key=score_of)
good.sort(key=score_of)
picked = []
for position in defective[-2:]:
    picked.append(("hit", position))
for position in defective[:4]:
    picked.append(("hardest defective", position))
for position in good[-4:]:
    picked.append(("hardest good", position))
draw_gallery(picked, "gallery_failures_" + CONFIG_ID + "_" + PRODUCT + ".png",
             "MVTec AD 2, " + PRODUCT + " (CC BY-NC-SA 4.0), half A, " + CONFIG_ID
             + ", seed " + str(SEED) + ". Red: mask. Darker blue: more suspicious")
with open(RESULTS_FOLDER / (CONFIG_ID + "_" + PRODUCT + "_failure_picks.csv"), "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["config", "product", "kind", "label", "file_name", "score"])
    for kind, position in picked:
        path = Path(paths[position])
        writer.writerow([CONFIG_ID, PRODUCT, kind, path.parent.name, path.name, round(float(scores[position]), 6)])
print("Finished", RUN_NAME)